# Incremental Silver Fact Sales — Delta MERGE

Repeatable notebook for the **sample dataset only**. Requires existing `silver.fact_sale` (produced by notebook 01) and the incremental Bronze Parquet files. The source is explicitly treated as authoritative for changed records **by laboratory convention**, not by an independently proven CDC event order.

Checks source uniqueness, counts and schema before writing. Updates only changed business columns and inserts missing keys. This notebook does not delete target rows absent from the source. Runs on a shared capacity can be expensive; coordinate with administrators and do not run concurrent writes against the same Delta table.

In [ ]:
from pyspark.sql import functions as F

TABLE_NAME = "silver.fact_sale"
BASE_PATH = "Files/bronze/wwi-raw-data"
MONEY_COLUMNS = ("TotalExcludingTax", "TaxAmount", "TotalIncludingTax")

def normalized_sales(path):
    df = spark.read.parquet(path)
    for name in MONEY_COLUMNS:
        df = df.withColumn(name, F.col(name).cast("decimal(38,6)"))
    return df

def check_keys(df, expected_count=None):
    result = df.agg(
        F.count("*").alias("rows"),
        F.countDistinct("SaleKey").alias("distinct_keys"),
        F.sum(F.col("SaleKey").isNull().cast("long")).alias("null_keys"),
        F.sum(F.col("InvoiceDateKey").isNull().cast("long")).alias("null_dates"),
        F.sum(F.col("Quantity").isNull().cast("long")).alias("null_quantities"),
        F.sum((F.col("Quantity") <= 0).cast("long")).alias("nonpositive_quantities"),
        *[F.sum(F.col(c).isNull().cast("long")).alias(f"null_{c}") for c in MONEY_COLUMNS]
    ).first()
    print(result.asDict())
    if expected_count is not None and result["rows"] != expected_count:
        raise ValueError(f"Unexpected row count: {result['rows']}")
    if result["rows"] != result["distinct_keys"] or result["null_keys"]:
        raise ValueError("SaleKey must be unique and not null")
    for name in ("null_dates", "null_quantities", "nonpositive_quantities", *(f"null_{c}" for c in MONEY_COLUMNS)):
        if result[name]:
            raise ValueError(f"Data-quality check failed: {name}={result[name]}")


In [ ]:
EXPECTED_INCREMENTAL_ROWS = 13611644
if not spark.catalog.tableExists(TABLE_NAME):
    raise RuntimeError("Initialize the historical Silver table first")

source = normalized_sales(f"{BASE_PATH}/incremental/fact_sale_1y_incremental")
check_keys(source, EXPECTED_INCREMENTAL_ROWS)

target_df = spark.table(TABLE_NAME)
expected_business_columns = [c for c in target_df.columns if not c.startswith("_")]
if source.columns != expected_business_columns:
    raise RuntimeError("Incremental schema does not match target business columns")
if any(source.schema[c].dataType != target_df.schema[c].dataType for c in source.columns):
    raise RuntimeError("Incremental types do not match Silver types")
print("Incremental preconditions passed")


In [ ]:
from delta.tables import DeltaTable

business_columns = [c for c in source.columns if c != "SaleKey"]
change_predicate = " OR ".join(
    f"NOT (s.`{c}` <=> t.`{c}`)" for c in business_columns
)
update_set = {c: f"s.`{c}`" for c in business_columns}
update_set.update({
    "_ingestion_timestamp": "current_timestamp()",
    "_record_source": "'incremental'",
})
insert_set = {c: f"s.`{c}`" for c in source.columns}
insert_set.update({
    "_ingestion_timestamp": "current_timestamp()",
    "_record_source": "'incremental'",
})

print("Latest version before merge:")
spark.sql(f"DESCRIBE HISTORY {TABLE_NAME}").select("version", "operation").show(1)
(
    DeltaTable.forName(spark, TABLE_NAME).alias("t")
    .merge(source.alias("s"), "t.SaleKey = s.SaleKey")
    .whenMatchedUpdate(condition=change_predicate, set=update_set)
    .whenNotMatchedInsert(values=insert_set)
    .execute()
)
print("MERGE completed")


In [ ]:
from functools import reduce
from operator import or_

EXPECTED_FINAL_ROWS = 54799826
final_table = spark.table(TABLE_NAME)
summary = final_table.agg(
    F.count("*").alias("rows"),
    F.countDistinct("SaleKey").alias("distinct_keys"),
    F.sum(F.col("SaleKey").isNull().cast("long")).alias("null_keys")
).first()
assert summary["rows"] == EXPECTED_FINAL_ROWS
assert summary["distinct_keys"] == EXPECTED_FINAL_ROWS
assert summary["null_keys"] == 0

s = source.alias("s")
t = final_table.alias("t")
changed = reduce(or_, [
    ~F.col(f"s.`{c}`").eqNullSafe(F.col(f"t.`{c}`"))
    for c in business_columns
])
joined = s.join(t, F.col("s.SaleKey") == F.col("t.SaleKey"), "left")
check = joined.agg(
    F.count("*").alias("source_rows"),
    F.sum(F.col("t.SaleKey").isNull().cast("long")).alias("missing_keys"),
    F.sum((F.col("t.SaleKey").isNotNull() & changed).cast("long")).alias("changed_rows")
).first()
assert check["missing_keys"] == 0
assert check["changed_rows"] == 0
print("Target reconciliation passed:", check.asDict())
spark.sql(f"DESCRIBE HISTORY {TABLE_NAME}").select("version", "operation", "operationMetrics").show(5, truncate=False)
